# CT.M1 - Data Acquisition
**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/01_pubchem_data_retrieval.ipynb)


---


### Contents

This notebook introduces the first stage of the computational workflow: retrieving molecular data from PubChem using its REST API.

The activities are structured to guide you through the complete process of:

1. Understanding the PubChem data model  
2. Querying compounds programmatically  
3. Retrieving molecular properties  
4. Constructing a structured dataset  
5. Performing initial data inspection  
6. Exporting a raw dataset for downstream curation  

This notebook is designed to be completed in approximately **45–60 minutes**.  

You are encouraged to modify queries, explore additional properties, and inspect returned data structures. The goal is not only to retrieve molecules, but to understand how chemical information is organized and accessed computationally.

---

### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourName_YourID_01_pubchem_data_retrieval.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

### PubChem Data Retrieval  

**PubChem** is a public chemical information resource maintained by the NIH, providing access to molecular structures, identifiers, and associated chemical properties.

In this notebook, we will **search PubChem**, **retrieve molecular properties programmatically**, and **construct a structured dataset** suitable for downstream curation and analysis.

---

### Step 0: The Computational Environment

Before retrieving chemical data, we must load our specialized toolkit. 
In this module, we rely on a standard Python stack for web communication and data manipulation:

* **`requests`**: A widely used library for making HTTP requests. It allows our script to cominicate to the PubChem PUG-REST API.
* **`pandas`**: A powerful library for structured data (DataFrames) manipulation. It will serve as our digital lab notebook to organize molecular properties.
* **`os` & `time`**: Utilities for file system management and for respecting API rate limits (avoiding excessive server requests).


>**Note:** While **RDKit** is central to structural cheminformatics workflows, we deliberately keep this first step lightweight. The goal here is to focus exclusively on raw data acquisition before moving into molecular representation and structural analysis.

In [ ]:
import os
import time
import requests
import pandas as pd


print("Current working directory:", os.getcwd())
print("Environment ready.")
print("requests version:", requests.__version__)

response = requests.get("https://pubchem.ncbi.nlm.nih.gov")
print("Connection status:", response.status_code)

### Step 1: Defining the Chemical Search Space
In this section, we define the search parameters. 

In a real-world project, this step represents the **initial library selection phase**, where researchers determine which region of chemical space they want to explore.
Here, we perform a name-based query in PubChem to retrive compounds associated with a given term (e.g., "flavonoid").

To keep the tutorial computationally manageable, we deliberately limit the number of retrieved compounds. This ensures faster execution while preserving the logic of large-scale data acquisition workflows.

In [ ]:
# Step 1

BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
OUT_DIR = os.path.join("..", "data_raw")
os.makedirs(OUT_DIR, exist_ok=True)

# Search configuration
QUERY = "flavonoid"   # Modify to explore other terms
MAX_CIDS = 200        # Limited size for tutorial efficiency

cids = get_cids_by_name(QUERY, MAX_CIDS, name_type="word")
print("Total CIDs retrieved:", len(cids))
print("First 10:", cids[:10])

### Step 2: From Natural Language to Unique CIDs

Chemical names are often ambiguous, inconsistent, or redundant. 

To ensure data integrity and reproducibility, we convert our textual query into **PubChem Compound Identifiers (CIDs)**. 
A CID is a permanent numerical identifier assigned to a compound record within the PubChem database, maintained by the National Center for Biotechnology Information (NCBI, NIH).

Much like a unique ID in a registry, the CID allows us to reference a specific compound unambiguously within PubChem’s chemical information system.

In [ ]:
# Step 2
from urllib.parse import quote

def get_cids_by_name(query: str, max_cids: int = 200, name_type: str = "complete"):
    """
    Retrieve PubChem CIDs associated with a textual query.
    """
    q = quote(query)  # encodes spaces, special chars
    url = f"{BASE_URL}/compound/name/{q}/cids/JSON?name_type={name_type}"
    print("Requesting:", url)
    
    r = requests.get(url, timeout=30)

    # If PubChem finds nothing, it may return 404 NotFound
    if r.status_code == 404:
        print("[!] NotFound: no matches for this query with the selected name_type.")
        return []

    r.raise_for_status()
    
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    
    if not cids:
        print("No CIDs found for this query.")
    
    return cids[:max_cids]


print("Total CIDs retrieved:", len(cids))
print("First 10 CIDs:", cids[:10])

> **Notice:**  
> We retrieved only one CID for the query `"flavonoid"`.  
> This occurs because PubChem interprets the term as a compound name rather than as a structural or chemical class.
>
> Text-based searches in PubChem rely on name matching, not on chemical ontology or structural classification. As a result, broad chemical categories may not automatically return large compound libraries.

### Step 3: Molecular Digitalization & Property Selection

In this section, we retrieve structured molecular descriptors for our list of CIDs. 
In cheminformatics, these descriptors can later be assembled into a **feature vector**—a numerical representation of a molecule suitable for computational analysis.

We selected properties that are fundamental for **Drug-Likeness evaluation** (inspired by Lipinski's Rule of Five) and future **QSAR** (Quantitative Structure–Activity Relationship) modeling:

* **SMILES (Simplified Molecular Input Line Entry System):** A line notation like a typographical method using printable characters, for entering and representing molecules. SMILES is the complete string that includes connectivity, stereochemical, and isotopic information. This is equivalent to the older *isomeric SMILES* concept.
* **ConnectivitySMILES**: A connectivity-only (simplified) SMILES string (no stereochemistry or isotopes).  
* **InChIKey:** A fixed-length (27-character) hashed identifier derived from the full InChI string. It is widely used for **deduplication** and rapid compound matching across databases.
* **Molecular Weight (MW):** A descriptor of molecular size. It contributes to absorption, distribution, and elimination properties in drug development.
* **XLogP:** An algorithmic estimate of the octanol-water partition coefficient ($\log P$). It reflects **lipophilicity**, which influences membrane permeability and solubility.
* **TPSA (Topological Polar Surface Area):** The surface area associated with polar atoms (typically nitrogen and oxygen). It is commonly used as an indicator of **oral bioavailability** and blood–brain barrier permeability.

In [ ]:
# Step 3
def fetch_properties(cids, chunk_size=50):
    props = [
        "SMILES",
        "ConnectivitySMILES",  # Optional: can be used for better standardization.
        "InChIKey",
        "MolecularWeight",
        "XLogP",
        "TPSA",
    ]

    if not cids:
        print("[!] Warning: Empty CID list provided")
        return pd.DataFrame(columns=["CID"] + props)

    all_rows = []
    total_cids = len(cids)

    for i in range(0, total_cids, chunk_size):
        batch = cids[i : i + chunk_size]
        cid_str = ",".join(map(str, batch))
        props_str = ",".join(props)
        url = f"{BASE_URL}/compound/cid/{cid_str}/property/{props_str}/JSON"

        if i == 0:
            print("Example endpoint:", url)

        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()

            batch_rows = r.json().get("PropertyTable", {}).get("Properties", [])
            all_rows.extend(batch_rows)

            processed = min(i + chunk_size, total_cids)
            print(f"✓ Progress: processed {processed}/{total_cids} CIDs", end="\r")

        except requests.exceptions.RequestException as e:
            print(f"\n[!] Request error in batch starting at index {i}: {e}")
            continue

        except ValueError as e:
            print(f"\n[!] JSON parsing error in batch at index {i}: {e}")
            continue

        time.sleep(0.2)

    print("\n✓ Fetch complete")

    df = pd.DataFrame(all_rows)

    # Reorder columns nicely for the workshop (CID first)
    if not df.empty and "CID" in df.columns:
        ordered = ["CID"] + [c for c in props if c in df.columns]
        df = df[ordered]

        print("Unique CIDs retrieved:", df["CID"].nunique())
    else:
        print("[!] Warning: No data retrieved (empty DataFrame)")

    return df


df = fetch_properties(cids)
print("Final dataset shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

### Step 4: Initial Data Inspection & Quality Control (QC)

"Garbage In, Garbage Out." Before moving to any modeling or visualization, we perform a sanity check on the retrieved data.

This QC step focuses on two core tasks:

1. **Missing values (NaNs):** Are key fields available for most compounds (e.g., `SMILES`, `InChIKey`, and physicochemical descriptors)?
2. **Duplicates:** Do we have repeated compounds in our dataset? We can detect duplicates using **InChIKey** (and optionally SMILES).

Even high-authority databases can contain incomplete or noisy records, so early QC helps prevent downstream errors and misleading analyses.

In [ ]:
# Step 4
df.head()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df.duplicated(subset=["SMILES"]).sum()

### Step 5: Data Persistence and File Serialization

The final step in the acquisition phase is to export our DataFrame into a persistent format. 
We save the dataset as a `.csv` file in the `data_raw` directory.

This establishes a **data provenance checkpoint**: we preserve the original API-retrieved information before any structural cleaning, filtering, or transformation occurs in the next module.

Maintaining a clear separation between **Raw** and **Processed** data is a best practice in computational research. It ensures reproducibility, traceability, and auditability throughout the analytical workflow.

In [ ]:
# Step 5

if df.empty:
    print("[!] Warning: DataFrame is empty. Saving empty file.")

filename = f"pubchem_raw_{QUERY}.csv"
out_path = os.path.join(OUT_DIR, filename)

df.to_csv(out_path, index=False)

print(f"✓ File successfully saved to: {out_path}")
print(f"Dataset shape: {df.shape}")

### Summary and Next Steps

You have successfully built a raw chemical dataset directly from the PubChem cloud API.

**Current State:**  
We now have a `.csv` file containing:
- 2D structural representations (SMILES)
- Unique identifiers (InChIKey)
- Basic physicochemical descriptors (Molecular Weight, XLogP, TPSA)

**Next Module:**  
We will use **RDKit** to perform chemical curation, including salt removal, normalization, tautomer handling, and structural validation. These steps will prepare the dataset for downstream analysis and descriptor generation.

---


### Exercises: Exploring the Chemical Space

Now that you have the pipeline working, use it to investigate how chemistry changes across different functional or therapeutic groups.

> **Important:** The endpoint used in this notebook performs **text-based name/synonym matching** in PubChem.  
> Broad category terms (e.g., `"alkaloid"`, `"antibiotic"`) may return few results or even `404 NotFound` when using strict matching.  
> If that happens, switch to `name_type="word"` (use for partial matching) and try again.

---

#### 1. Comparative Analysis
Change the `QUERY` variable to different terms and observe the results. Examples:

* **Specific molecules (usually reliable with `name_type="complete"`):**  
  `"aspirin"`, `"capsaicin"`, `"cannabidiol"`

* **Chemical-class terms (often require `name_type="word"`):**  
  `"alkaloid"`, `"flavanone"`, `"terpene"`

* **Pharmacological group terms (usually require `name_type="word"`):**  
  `"kinase inhibitor"`, `"antibiotic"`, `"antiviral"`

Re-run the notebook and compare dataset size and descriptor availability.

---

#### 2. Data Integrity Challenge
After running a query that yields a larger dataset (e.g., `"antibiotic"` with `MAX_CIDS = 1000` and `name_type="word"`), check:

* How many molecules have missing (**NaN**) `XLogP`?
* Why might some compounds lack `XLogP` while still having `MolecularWeight`?
* *Hint: Consider calculated vs. unavailable properties, salts/mixtures, and unusual structures.*

---

#### 3. The Representation Challenge: SMILES vs. ConnectivitySMILES
Modify the `props` list in your function to include both `"SMILES"` and `"ConnectivitySMILES"`. 
* Pick a chiral molecule like **"Taxol"** or **"Glucose"**.
* **Compare the strings:** What are the extra characters in the `SMILES` (the default *isomeric* `SMILES` in PubChem)? 
* **Reflect:** Why would using only `ConnectivitySMILES` be dangerous when designing a drug that must fit into a specific protein binding site?

---

#### 4. Critical Thinking: Physicochemical Profiling
Compare the **Molecular Weight** and **TPSA** of a *specific molecule* (`"aspirin"`) vs. a *broader pharmacological group* (`"kinase inhibitor"` with `name_type="word"`).

* What does the difference in TPSA suggest about membrane permeability (and possibly BBB penetration)?
* Based on **XLogP**, which query returned the most lipophilic (“greasy”) compounds?

---

#### 5. Advanced (extra points)
Modify the `props` list inside `fetch_properties()` to include:

- `"HBondDonorCount"`
- `"HBondAcceptorCount"`

Re-run the acquisition. Do these counts correlate with **TPSA**? Why would you expect a relationship?